# Notebook 07 — Upgraded Process Fidelity and Phase Diagnostics

This notebook extends the CZ-inspired pulse-sequence benchmark with **explicit phase diagnostics**.

Instead of only computing raw process fidelity against

$$
U_{\mathrm{CZ}} = \mathrm{diag}(1,1,1,-1),
$$

we also measure the **controlled phase** induced by the effective operation.

For a nearly diagonal effective unitary with diagonal phases
$\phi_{00}, \phi_{01}, \phi_{10}, \phi_{11}$,
the entangling phase is

$$
\phi_{\mathrm{ent}} = \phi_{00} + \phi_{11} - \phi_{01} - \phi_{10} \pmod{2\pi}.
$$

A CZ gate ideally has

$$
\phi_{\mathrm{ent}} = \pi.
$$

This notebook reports:
- raw process fidelity,
- unitarity error,
- entangling phase,
- a phase-corrected CZ comparison.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, sigmay, sesolve

## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
sy = sigmay()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
sy1 = tensor(sy, I)
sy2 = tensor(I, sy)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Hamiltonian and pulse-sequence helpers

In [ ]:
def two_atom_hamiltonian(omega1: float, omega2: float, delta: float, V: float, axis: str = 'x'):
    if axis == 'x':
        drive = 0.5 * omega1 * sx1 + 0.5 * omega2 * sx2
    elif axis == 'y':
        drive = 0.5 * omega1 * sy1 + 0.5 * omega2 * sy2
    else:
        raise ValueError("axis must be 'x' or 'y'")

    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def run_segment(state, omega1: float, omega2: float, delta: float, V: float,
                duration: float, axis: str = 'x', n_steps: int = 160):
    H = two_atom_hamiltonian(omega1=omega1, omega2=omega2, delta=delta, V=V, axis=axis)
    times = np.linspace(0.0, duration, n_steps)
    result = sesolve(H, state, times)
    return result.states[-1], times, result.states


def run_minimal_sequence(psi0, omega: float, delta: float, V: float,
                         t_pi: float, t_mid: float,
                         axis1: str = 'x', axis2: str = 'x', axis3: str = 'x'):
    state1, times1, states1 = run_segment(
        psi0, omega1=omega, omega2=0.0, delta=delta, V=V,
        duration=t_pi, axis=axis1
    )
    state2, times2, states2 = run_segment(
        state1, omega1=0.0, omega2=omega, delta=delta, V=V,
        duration=t_mid, axis=axis2
    )
    state3, times3, states3 = run_segment(
        state2, omega1=omega, omega2=0.0, delta=delta, V=V,
        duration=t_pi, axis=axis3
    )

    full_times = np.concatenate([
        times1,
        times2 + times1[-1],
        times3 + times1[-1] + times2[-1],
    ])
    full_states = states1 + states2 + states3
    return full_times, full_states, state3


def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)


def build_effective_unitary(omega: float, delta: float, V: float, t_pi: float, t_mid: float,
                            axis1: str = 'x', axis2: str = 'x', axis3: str = 'x'):
    columns = []
    final_states = []
    for psi0 in basis_states:
        _, _, psi_final = run_minimal_sequence(
            psi0,
            omega=omega,
            delta=delta,
            V=V,
            t_pi=t_pi,
            t_mid=t_mid,
            axis1=axis1,
            axis2=axis2,
            axis3=axis3,
        )
        columns.append(state_to_column(psi_final))
        final_states.append(psi_final)

    U_eff = np.column_stack(columns)
    return U_eff, final_states


## Gate metrics and phase diagnostics

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))


def unitarity_error(U_eff):
    identity = np.eye(U_eff.shape[0], dtype=complex)
    return float(np.linalg.norm(np.conjugate(U_eff.T) @ U_eff - identity))


def wrapped_phase(x):
    return np.angle(np.exp(1j * x))


def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))


def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    phi_ent = phi[0] + phi[3] - phi[1] - phi[2]
    return wrapped_phase(phi_ent)


def phase_corrected_cz_target(U_eff):
    phi = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    # remove single-qubit-like diagonal phases while preserving entangling phase
    # target = diag(1,1,1,e^{i phi_ent})
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)


def phase_corrected_fidelity(U_eff):
    U_target = phase_corrected_cz_target(U_eff)
    return process_fidelity(U_eff, U_target=U_target)


def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))


## Evaluate one operating point

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0
t_pi = np.pi / omega
t_mid = 2 * np.pi / omega

U_eff_xxx, _ = build_effective_unitary(
    omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid,
    axis1='x', axis2='x', axis3='x'
)
U_eff_xyx, _ = build_effective_unitary(
    omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid,
    axis1='x', axis2='y', axis3='x'
)

for label, U_eff in [('x-x-x', U_eff_xxx), ('x-y-x', U_eff_xyx)]:
    print(label)
    print(f'  raw process fidelity:        {process_fidelity(U_eff):.6f}')
    print(f'  phase-corrected fidelity:    {phase_corrected_fidelity(U_eff):.6f}')
    print(f'  entangling phase (rad):      {entangling_phase(U_eff):.6f}')
    print(f'  entangling phase / pi:       {entangling_phase(U_eff)/np.pi:.6f}')
    print(f'  unitarity error:             {unitarity_error(U_eff):.6e}')
    print(f'  off-diagonal leakage norm:   {leakage_metric(U_eff):.6e}')
    print()

U_eff = U_eff_xyx


## Visualize the effective gate matrix magnitude

In [ ]:
plt.figure(figsize=(6, 5))
im = plt.imshow(np.abs(U_eff), origin='lower', aspect='equal')
plt.xticks(range(4), basis_labels)
plt.yticks(range(4), basis_labels)
plt.xlabel('Input basis state')
plt.ylabel('Output basis amplitude')
plt.title(r'$|U_{eff}|$ for the upgraded pulse sequence')
plt.colorbar(im, label='magnitude')
plt.tight_layout()
plt.show()

## Scan the middle pulse duration

In [ ]:
t_mid_vals = np.linspace(0.2 * np.pi / omega, 3.2 * np.pi / omega, 50)
raw_xxx = []
raw_xyx = []
pc_xxx = []
pc_xyx = []
phi_xxx = []
phi_xyx = []

for t_mid_test in t_mid_vals:
    U_xxx, _ = build_effective_unitary(
        omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid_test,
        axis1='x', axis2='x', axis3='x'
    )
    U_xyx, _ = build_effective_unitary(
        omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid_test,
        axis1='x', axis2='y', axis3='x'
    )
    raw_xxx.append(process_fidelity(U_xxx))
    raw_xyx.append(process_fidelity(U_xyx))
    pc_xxx.append(phase_corrected_fidelity(U_xxx))
    pc_xyx.append(phase_corrected_fidelity(U_xyx))
    phi_xxx.append(entangling_phase(U_xxx))
    phi_xyx.append(entangling_phase(U_xyx))


In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, raw_xxx, label='raw x-x-x')
plt.plot(t_mid_vals, raw_xyx, label='raw x-y-x')
plt.axvline(2 * np.pi / omega, linestyle='--', label=r'$2\pi/\Omega$')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel(r'Raw $F_{proc}$')
plt.title('Raw process fidelity versus middle pulse duration')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, pc_xxx, label='phase-corrected x-x-x')
plt.plot(t_mid_vals, pc_xyx, label='phase-corrected x-y-x')
plt.axvline(2 * np.pi / omega, linestyle='--', label=r'$2\pi/\Omega$')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel('Phase-corrected fidelity')
plt.title('Phase-corrected fidelity versus middle pulse duration')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, np.array(phi_xxx) / np.pi, label='x-x-x')
plt.plot(t_mid_vals, np.array(phi_xyx) / np.pi, label='x-y-x')
plt.axhline(1.0, linestyle='--', label=r'ideal $\pi$ phase')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel(r'Entangling phase / $\pi$')
plt.title('Entangling phase versus middle pulse duration')
plt.legend()
plt.tight_layout()
plt.show()

## Process-fidelity and phase landscape over $(\Omega, V)$

In [ ]:
omega_vals = np.linspace(0.4, 2.0, 24)
V_vals = np.linspace(0.0, 8.0, 24)

raw_landscape = np.zeros((len(V_vals), len(omega_vals)))
pc_landscape = np.zeros((len(V_vals), len(omega_vals)))
phi_landscape = np.zeros((len(V_vals), len(omega_vals)))
leak_landscape = np.zeros((len(V_vals), len(omega_vals)))

for i, V_scan in enumerate(V_vals):
    for j, omega_scan in enumerate(omega_vals):
        t_pi_scan = np.pi / omega_scan
        t_mid_scan = 2 * np.pi / omega_scan
        U_scan, _ = build_effective_unitary(
            omega=omega_scan,
            delta=0.0,
            V=V_scan,
            t_pi=t_pi_scan,
            t_mid=t_mid_scan,
            axis1='x', axis2='y', axis3='x'
        )
        raw_landscape[i, j] = process_fidelity(U_scan)
        pc_landscape[i, j] = phase_corrected_fidelity(U_scan)
        phi_landscape[i, j] = entangling_phase(U_scan) / np.pi
        leak_landscape[i, j] = leakage_metric(U_scan)


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(raw_landscape, origin='lower', aspect='auto', extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]])
plt.contour(omega_vals, V_vals, raw_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Raw process fidelity over $(\Omega, V)$ for x-y-x sequence')
plt.colorbar(im, label=r'raw $F_{proc}$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_landscape, origin='lower', aspect='auto', extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]])
plt.contour(omega_vals, V_vals, pc_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Phase-corrected fidelity over $(\Omega, V)$ for x-y-x sequence')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(phi_landscape, origin='lower', aspect='auto', extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]], vmin=-1, vmax=1)
plt.contour(omega_vals, V_vals, phi_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Entangling phase / $\pi$ over $(\Omega, V)$ for x-y-x sequence')
plt.colorbar(im, label=r'$\phi_{ent}/\pi$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(leak_landscape, origin='lower', aspect='auto', extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]])
plt.contour(omega_vals, V_vals, leak_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Off-diagonal leakage norm over $(\Omega, V)$ for x-y-x sequence')
plt.colorbar(im, label='leakage norm')
plt.tight_layout()
plt.show()

## Best points in the $(\Omega, V)$ scan

In [ ]:
best_raw_idx = np.unravel_index(np.argmax(raw_landscape), raw_landscape.shape)
best_pc_idx = np.unravel_index(np.argmax(pc_landscape), pc_landscape.shape)

print('Best raw process-fidelity point:')
print(f'  Omega = {omega_vals[best_raw_idx[1]]:.3f}, V = {V_vals[best_raw_idx[0]]:.3f}, raw F = {raw_landscape[best_raw_idx]:.6f}, phase-corrected F = {pc_landscape[best_raw_idx]:.6f}, phi_ent/pi = {phi_landscape[best_raw_idx]:.6f}')
print()
print('Best phase-corrected-fidelity point:')
print(f'  Omega = {omega_vals[best_pc_idx[1]]:.3f}, V = {V_vals[best_pc_idx[0]]:.3f}, raw F = {raw_landscape[best_pc_idx]:.6f}, phase-corrected F = {pc_landscape[best_pc_idx]:.6f}, phi_ent/pi = {phi_landscape[best_pc_idx]:.6f}')

## Interpretation

This upgrade separates three different failure modes:

- **raw process fidelity** can be low when diagonal phases are wrong,
- **phase-corrected fidelity** measures whether the operation is close to a controlled-phase gate up to simple diagonal phase freedom,
- **entangling phase** checks whether the two-body phase is actually approaching the CZ target of $\pi$,
- **leakage norm** reveals whether non-diagonal mixing remains even if phases look promising.

This is more informative than raw process fidelity alone because it distinguishes:
1. wrong entangling phase,
2. correct entangling phase but wrong local/global phases,
3. residual leakage out of the ideal diagonal structure.


## Optional next steps

Natural next upgrades are:

- tune $t_{mid}$ to drive $\phi_{ent} \rightarrow \pi$,
- add detuning ramps during the middle pulse,
- include explicit single-qubit phase corrections,
- optimize over $(\Omega, V, t_{mid}, \Delta)$ to maximize the phase-corrected fidelity while minimizing leakage.
